In [1]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
import pandas as pd
import time
import numpy as np

#Import data
csvfile = 'marketing_campaign_cleaned_NODUMMIES'
df = pd.read_csv(csvfile)

#Imports from synthcity package
import sys
import warnings

warnings.filterwarnings("ignore")

import synthcity.logger as log
from synthcity.plugins import Plugins
from synthcity.plugins.core.dataloader import GenericDataLoader

log.add(sink=sys.stderr, level="INFO")
syn_model_full = Plugins().get("ctgan")

##########################FUNCTIONS#################################
def dummify_columns(df):
    
    dummify_marital = pd.get_dummies(df['Marital_Status'],prefix='marital')
    df = pd.concat([df, dummify_marital],axis=1)

    dummify_edu = pd.get_dummies(df['Education'],prefix='education')
    df = pd.concat([df, dummify_edu], axis=1)

    df.drop(columns=['Marital_Status', 'Education'], inplace=True)
    
    return df

def train_and_evaluate(df_train, df_test):

    X_train = df_train.drop('Response', axis=1)
    y_train = df_train['Response']
    X_test = df_test.drop('Response', axis=1)
    y_test = df_test['Response']

    # Identify overlapping columns in train and test: note that the train set is only small (7.5p), columns may be missing
    # if we add only a little bit of synth data, there is a chance that same column is missing in synth data
    common_columns = set(X_train.columns).intersection(X_test.columns)
    
    # Keep only common columns in train and test
    X_train = X_train[common_columns]
    X_test = X_test[common_columns]
    
    # RF model
    model = RandomForestClassifier(n_estimators = 500, max_depth = None, 
                                   max_features = 'auto', criterion = 'gini', min_samples_split = 2)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # Metrics
    report = classification_report(y_test, y_pred, output_dict=True)
    roc_auc = roc_auc_score(y_test, y_prob)

    results = {
        'accuracy': report['accuracy'],
        'precision': report['1']['precision'],
        'recall': report['1']['recall'],
        'f1': report['1']['f1-score'],
        'auc-roc': roc_auc
    }

    return results
##################################################################





<stdin>:1:10: fatal error: 'omp.h' file not found
#include <omp.h>
         ^~~~~~~
1 error generated.


[KeOps] Warning : omp.h header is not in the path, disabling OpenMP.
[KeOps] Warning : Cuda libraries were not detected on the system ; using cpu only mode
2023-08-07 21:48:25,901 - Created a temporary directory at /var/folders/8w/flfvck1j6m77x3j6jf0384bw0000gn/T/tmpw1s9gs2s
2023-08-07 21:48:25,902 - Writing /var/folders/8w/flfvck1j6m77x3j6jf0384bw0000gn/T/tmpw1s9gs2s/_remote_module_non_scriptable.py


In [2]:
# Bootstrap

#Synthetic data sizes generated

syn_sizes = [1, 0.5*(len(df)*0.7) , 1*(len(df)*0.7), 3*(len(df)*0.7) , 
             5*(len(df)*0.7) , 8*(len(df)*0.7) , 12*(len(df)*0.7) , 
             18*(len(df)*0.7) , 32*(len(df)*0.7) , 48*(len(df)*0.7), 64*(len(df)*0.7)]  

n_iterations = 300  # Number of bootstrapping iterations
syn_model = Plugins().get('ctgan')

results = []

# Bootstrap iteration loop
for i in range(n_iterations):
    
    start_time = time.time()

    #Entire dataset is resampled
    df_resampled = resample(df, replace=True)

    #Do train-test split
    df_train, df_test = train_test_split(df_resampled, test_size=0.3, 
                                         stratify=df_resampled['Response'], random_state=i)

    #Load and fit data to CTGAN. Data preprocessing is partially doen inside CTGAN program
    
    loader = GenericDataLoader(df_train, target_column='Response')
    syn_model.fit(loader,cond=df_train['Response'].to_frame())
    
    # Loop through synth set sizes
    for size in syn_sizes:
        #Generate synth. data. Note that a condition is provided to enforce 85/15 split in target var.
        syn_set = syn_model.generate(count=size,random_state=i,
                                     cond=np.random.permutation([1]*int(round(df['Response'].mean()*size)) 
                                                                + [0]*int(size-round(df['Response'].mean()*size)))).dataframe()
        
        # Combine real and synth data
        df_train_combined = pd.concat([df_train, syn_set], axis=0)

        # Dummify train and test datasets to feed to RF (no dummification beforehand to not interfer with CTGAN internal process)
        df_train_combined = dummify_columns(df_train_combined)
        df_test_dummified = dummify_columns(df_test.copy())  # .copy() to avoid SettingWithCopyWarning

        # Train and evaluate
        metrics = train_and_evaluate(df_train_combined, df_test_dummified)

        # Store the results with additional information
        metrics['syn_size'] = size
        metrics['iteration'] = i
        results.append(metrics)

    end_time = time.time()
    execution_time = end_time - start_time
    print(f"Time: {execution_time} seconds")
    print(f"Iteration: {i} " )
    
# Convert to DataFrame
results_df = pd.DataFrame(results)

 22%|████████▉                               | 449/2000 [02:16<07:51,  3.29it/s]


Time: 376.30721282958984 seconds
Iteration: 0 


 47%|██████████████████▉                     | 949/2000 [05:25<06:00,  2.91it/s]


Time: 574.1103889942169 seconds
Iteration: 1 


 77%|██████████████████████████████▏        | 1549/2000 [08:28<02:28,  3.05it/s]


...

Time: 397.10488080978394 seconds
Iteration: 109 


 22%|████████▉                               | 449/2000 [02:19<08:02,  3.21it/s]


Time: 374.6381709575653 seconds
Iteration: 110 


 40%|███████████████▉                        | 799/2000 [04:28<06:44,  2.97it/s]


KeyboardInterrupt: 

In [ ]:
results_df.to_clipboard()